# RS07 · Behavior trees for robot tasks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/07_reliability_systems/RS07_behavior_trees_for_robot_tasks.ipynb)

**Track:** Reliability and systems · **Skill axis (sheet):** Reliability and systems · **Time:** about 50 min

**Skill Hub concepts:** Behavior tree

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("RS07")

## Map: a tree ticked many times per second

Every tick starts at the root; each node returns **SUCCESS**, **FAILURE** or **RUNNING**.

| Node | Rule |
|---|---|
| Condition | SUCCESS if true, FAILURE if false (never RUNNING) |
| Action | does work; RUNNING while in progress |
| Sequence (reactive) | tick children left to right; return the first non-SUCCESS; all SUCCESS -> SUCCESS |
| Fallback (selector) | tick children left to right; return the first non-FAILURE; all FAILURE -> FAILURE |
| Retry(n) decorator | on child FAILURE try again, up to n failures in total, then FAILURE |

```
 Fallback
 ├── Condition: object in gripper?                        <- already done? then SUCCESS
 └── Sequence
     ├── Condition: object visible?                       <- re-checked every tick (reactive)
     ├── Action: approach                                 <- RUNNING for a few ticks
     └── Retry(3)
         └── Action: grasp
```

| | Behavior tree | Hierarchical state machine |
|---|---|---|
| Reacting to a changed condition | automatic: conditions are re-evaluated every tick | needs explicit transitions from every state |
| Adding a recovery behavior | insert a Fallback branch | add states and many transitions |
| Strong at | task logic, retries, priorities (Nav2, game AI) | modes with clear lifecycles (drives, safety, lab RS06) |

In [ ]:
SUCCESS, FAILURE, RUNNING = "SUCCESS", "FAILURE", "RUNNING"

class Condition:
    def __init__(self, fn): self.fn = fn
    def tick(self): return SUCCESS if self.fn() else FAILURE

class Action:
    def __init__(self, fn): self.fn = fn
    def tick(self): return self.fn()

## Task 1: the composite and decorator nodes

In [ ]:
class Sequence:
    def __init__(self, children): self.children = children
    def tick(self):
        raise NotImplementedError  # TODO

class Fallback:
    def __init__(self, children): self.children = children
    def tick(self):
        raise NotImplementedError  # TODO

class Retry:
    def __init__(self, child, n): self.child, self.n, self.failures = child, n, 0
    def tick(self):
        raise NotImplementedError  # TODO

lab.check("nodes", Sequence, Fallback, Retry)

In [ ]:
%%writefile rs07_nodes.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/rs07.h
// Grade it with lab.check("nodes", "rs07_nodes.c"). For C++, name the file .cpp in both places.
#include "rs07.h"

struct BtNode {
    int kind;  /* TODO: leaf callback, children, retry counters */
};

BtNode *bt_leaf(BtLeafFn fn, void *ctx) { (void)fn; (void)ctx; return NULL; /* TODO */ }
BtNode *bt_sequence(BtNode **children, size_t n) { (void)children; (void)n; return NULL; /* TODO */ }
BtNode *bt_fallback(BtNode **children, size_t n) { (void)children; (void)n; return NULL; /* TODO */ }
BtNode *bt_retry(BtNode *child, int n) { (void)child; (void)n; return NULL; /* TODO */ }
int bt_tick(BtNode *node) { (void)node; return BT_FAILURE; /* TODO */ }

In [ ]:
# lab.check("nodes", "rs07_nodes.c")  # uncomment after writing the C/C++ cell above

## Task 2: run the pick tree against a tiny world

In [ ]:
class World:
    def __init__(self, grasp_fail_first=1):
        self.visible, self.in_gripper, self.distance = True, False, 3
        self.grasp_fail_first, self.grasp_attempts = grasp_fail_first, 0
    def approach(self):
        if self.distance > 0:
            self.distance -= 1
            return RUNNING
        return SUCCESS
    def grasp(self):
        self.grasp_attempts += 1
        if self.grasp_attempts <= self.grasp_fail_first:
            return FAILURE
        self.in_gripper = True
        return SUCCESS

def build_tree(w):
    raise NotImplementedError  # TODO

lab.check("tree", build_tree, World)

In [ ]:
try:
    w = World(grasp_fail_first=1); tree = build_tree(w)
    for k in range(8):
        print(k, tree.tick(), "distance", w.distance, "grasps", w.grasp_attempts, "holding", w.in_gripper)
except NotImplementedError:
    print("finish the tasks first")

## Checkpoint: Math
**M1 (Apply).** A Fallback of three conditions, each taking 2 ms to evaluate. Worst-case time for one tick, in ms?

**M2 (Apply).** The tree ticks at 10 Hz. Worst-case delay (ms) before it reacts to the object disappearing?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Analyze).** Mid-approach, the camera loses the object. What does the reactive sequence do on the next tick?
- a) finishes the approach first
- b) the visibility condition fails, so the sequence returns FAILURE right away and approach is not ticked again
- c) retries the grasp

**K2 (Evaluate).** For Gambit's pick-and-place, which parts belong in a behavior tree and which in a state machine, and why?

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `rs07.h`).
3. In `roboprep/checks/rs07.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()